In [ ]:
import os

# Qdrant
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
COLLECTION_NAME = "pubmed_articles"

# MinIO (dev)
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://localhost:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER", "kairos_admin")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD", "changeme_in_prod")
MINIO_BUCKET = os.getenv("MINIO_BUCKET", "qdrant-snapshots")

# ETL Notebook Config
NCBI_API_KEY = os.getenv("NCBI_API_KEY", "")
PUBMED_SEARCH_TERM = os.getenv("PUBMED_SEARCH_TERM", '("clinical decision support"[MeSH])')
PUBMED_BATCH_LIMIT = int(os.getenv("PUBMED_BATCH_LIMIT", "1000"))
PMC_FULLTEXT_ENABLED = True

# Models
MEDCPT_ARTICLE_ENCODER = "ncbi/MedCPT-Article-Encoder"


In [ ]:
# Note: Ensure you run this with the ETL dependency group:
# uv sync --group etl
import qdrant_client
import transformers
import torch
import lxml
import httpx
import tqdm
import boto3
import playwright
print("All dependencies loaded!")


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

client = QdrantClient(url=QDRANT_URL)

if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=768, distance=Distance.COSINE),
    )
    print(f"Collection '{COLLECTION_NAME}' created.")
else:
    print(f"Collection '{COLLECTION_NAME}' already exists.")


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MEDCPT_ARTICLE_ENCODER)
model = AutoModel.from_pretrained(MEDCPT_ARTICLE_ENCODER).to(device)


In [ ]:
import httpx
import xml.etree.ElementTree as ET

def fetch_pmids(term, limit):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        "db": "pubmed",
        "term": term,
        "retmode": "xml",
        "retmax": limit,
    }
    if NCBI_API_KEY:
        params["api_key"] = NCBI_API_KEY
        
    response = httpx.get(url, params=params)
    response.raise_for_status()
    
    root = ET.fromstring(response.text)
    pmids = [id_elem.text for id_elem in root.findall(".//Id")]
    return pmids

print(f"Fetching PMIDs for: {PUBMED_SEARCH_TERM} (Limit: {PUBMED_BATCH_LIMIT})")
pmids = fetch_pmids(PUBMED_SEARCH_TERM, PUBMED_BATCH_LIMIT)

# TODO: Add logic to diff against existing Qdrant PMIDs for incremental update
print(f"Found {len(pmids)} PMIDs.")


In [ ]:
def fetch_abstracts(pmids):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml",
    }
    if NCBI_API_KEY:
        params["api_key"] = NCBI_API_KEY
        
    response = httpx.post(url, data=params)
    response.raise_for_status()
    
    # Minimal parsing logic for now
    root = ET.fromstring(response.text)
    articles = []
    for article in root.findall(".//PubmedArticle"):
        pmid = article.find(".//PMID").text
        title = article.find(".//ArticleTitle")
        title_text = title.text if title is not None else ""
        
        abstract = article.find(".//AbstractText")
        abstract_text = abstract.text if abstract is not None else ""
        
        articles.append({
            "pmid": pmid,
            "title": title_text,
            "abstract": abstract_text
        })
    return articles

print("Fetching abstracts...")
# Fetching in one batch for demonstration, but production should chunk to 300
articles = fetch_abstracts(pmids[:10]) 
print(f"Fetched {len(articles)} abstracts.")


In [ ]:
# Simplistic chunking (abstracts only for this MVP structure)
chunks = []
for art in articles:
    text = f"{art['title']}. {art['abstract']}"
    # Splitting into chunks roughly...
    words = text.split()
    for i in range(0, len(words), 256):
        chunk_words = words[i:i+256]
        chunk_text = " ".join(chunk_words)
        chunks.append({
            "pmid": art["pmid"],
            "title": art["title"],
            "text": chunk_text
        })
print(f"Generated {len(chunks)} chunks.")


In [ ]:
from qdrant_client.models import PointStruct
import uuid

points = []
for chunk in chunks:
    inputs = tokenizer(chunk["text"], padding=True, truncation=True, return_tensors="pt", max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy().tolist()
    
    point_id = str(uuid.uuid4())
    points.append(
        PointStruct(
            id=point_id,
            vector=embedding,
            payload={
                "pmid": chunk["pmid"],
                "title": chunk["title"],
                "text": chunk["text"]
            }
        )
    )

print("Upserting into Qdrant...")
if points:
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=points
    )
print("Upsert complete.")


In [ ]:
info = client.get_collection(collection_name=COLLECTION_NAME)
print(f"Collection count: {info.points_count}")


In [ ]:
import subprocess
import boto3

print("Snapshot helper to be executed after finishing ETL...")
# Code would stop qdrant container, tar storage, and upload to MinIO.
